In [ ]:
%%capture

%pip install langchain-community -U
%pip install langchain-google-genai
%pip install pypdf
%pip install langchain
%pip install langchain-chroma
%pip install langchain -U
%pip install langchain_cohere

In [36]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.documents import Document

In [2]:
# Carregar e extrair pdf
loader = PyPDFLoader('os-sertoes.pdf')
documents = loader.load()

# Criar chuncks
text_splitter = CharacterTextSplitter(chunk_size=2048, chunk_overlap=128)
chunks = text_splitter.split_documents(documents)

In [3]:
# Embeddings
embeddings_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# Extrair o conteúdo de cada Document para gerar embeddings
text_contents = [doc.page_content for doc in chunks]
embeddings = embeddings_model.embed_documents(text_contents)

In [ ]:
# Criação do retriever para encontrar os chunks mais relevantes 
vectorstore = Chroma.from_documents(chunks, embedding=embeddings_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 7})

In [5]:
# Init modelo Gemini e criação do prompt template
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2)
prompt = ChatPromptTemplate.from_template("""
    Você é um bibliotecário. Responda as perguntas baseadas no contexto fornecido.
                                          
    Context: {context}
                                          
    Pergunta: {input}
""")

document_chain = create_stuff_documents_chain(llm, prompt)

In [ ]:
asks = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em \"Os Sertões\"? Como esses aspectos refletem a visão do autor sobre o Brasil da época?"
]
import cohere
for ask in asks:
    # Recupera os chunks mais relevantes baseado na pergunta
    docs = retriever.invoke(ask)
    co = cohere.ClientV2()
    page_contents = [doc.page_content for doc in docs]
    #Aplica o reranker para ranquear novamente os chunks mais relevantes
    context = co.rerank(
        model="rerank-v3.5", query=ask, documents=page_contents, top_n=3
    )
    
    # Criar novo contexto com os documentos com melhor ranqueamento.
    new_docs = []
    for c in context.results:
        new_docs.append(Document(docs[c.index].page_content))
    
    # Invoke da LLM com a pergunta e os documentos obtidos no RAG.
    response = document_chain.invoke({"input": ask, "context": new_docs})

    # Imprima a resposta
    print(f"Pergunta: {ask}\nResposta: {response}\n\n")

Pergunta: Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?
Resposta: Euclides da Cunha descreve o sertão nordestino como um ambiente hostil, marcado pela seca e pela falta de recursos hídricos.  Ele vê a extensa superfície de evaporação como um fator que contribui para a aridez da região, tornando a vida dos habitantes extremamente difícil.  Cisternas, poços artesianos e lagos esparsos são soluções locais e insuficientes para combater o problema central: o deserto.  Para Cunha, o sofrimento humano no sertão é um reflexo da "tortura maior" sofrida pela própria terra, um martírio secular que afeta a economia da vida na região.  A seca não é apenas uma questão de falta de água, mas a causa raiz de uma série de problemas que afetam profundamente a existência dos sertanejos.


Pergunta: Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas caracter